# 04 — Context-Aware (Synthetic Proof-of-Concept)

**Group 9 contributors**

- ANDREA BASTIEN SAXOD
- CLOÉ KARINA CHAPOTOT
- CONSTANTIN NICOLA HATECKE
- MARCELA MENDES GIMENES FUNABASHI
- MATIAS VICENTE AREVALO MARTINEZ
- VITTORIO FIALDINI

## Objective

Layer club context on top of player similarity. "Context" means three things: stylistic match between the player's preferred environment and the team's playing identity, gap fit between the player's on-ball contribution and the position where the club is weakest relative to its peers, and budget fit between the player's market value and the team's budget band.

## Inputs

- `players_master_synthetic_augmented.csv`: synthetic traits, preferences and contribution vectors at player level.
- `teams_master_synthetic_augmented.csv`: synthetic latent style, tactical archetype, and budget band at team level.
- `players_scored` from notebook 01: real-only quality score used as a fourth component.

## Transparency note, read before the shortlists

This notebook used to consume pre-computed CSV shortlists. The scoring formula was not visible in the five submitted notebooks, which made the context-aware layer opaque to a grader. We rebuild it here so every number is auditable.

**All synthetic layers used here are proof-of-concept scenario modelling, not an unbiased real-world benchmark.** The honest real-only benchmark is computed in `05_evaluation.ipynb` using only real FBref features and real transfer arrivals as ground truth. Treat this notebook as "how we would combine context if we had trustworthy tactical descriptors," not as "this is how good the model actually is."

## Method summary

The final score is a weighted combination of four sub-scores:

1. `style_fit_poc = 1 - mean(|player_pref - team_latent|)` across six paired axes: possession, directness, pressing, width, tempo, territorial dominance.
2. `need_fit_poc = dot(player_contrib, team_need_vector)` where `team_need_vector` is the unit-norm gap between the top-quartile peer profile at the position and the club's own current profile.
3. `budget_fit_poc = exp(-diff^2 / (2 * sigma^2))` where diff is the distance between the player's log-market-value band and the team's synthetic budget band.
4. `quality_logistic = sigmoid(real_quality_score)` from notebook 01, so strong performers are not penalised for being a stylistic middleground.

Combined:

    context_syn_poc_score =
        0.35 * style_fit_poc +
        0.30 * (0.5 + 0.5 * need_fit_poc) +
        0.20 * budget_fit_poc +
        0.15 * quality_logistic

Weights sum to one. The `need_fit_poc` term can be negative, which is why it is remapped into [0, 1] before weighting. Candidates are drawn from the previous season, filtered by position, and exclude players already at the target club.

## Key assumptions

- The synthetic layer's player preferences and team latent styles are comparable (same [0, 1] scale, paired by axis definition).
- The contribution vector is a reasonable five-dimensional proxy for role-specific on-ball output.
- Budget fit is symmetric around the team's band; underpriced players are not extra-rewarded.

## Outputs

- Team-archetype distribution in 2023-24, with sample rows.
- Audit: actual arrivals versus non-arrivals, mean sub-score gap by role. This is the "uplift audit" the synthetic layer is useful for.
- Shortlists for Arsenal (MF) and Girona (FW) with per-component explanation strings.
- An explicit statement, at the end, that all uplift numbers are scenario modelling not generalisation.


In [1]:
import warnings, json
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def locate_project_root():
    """Find a root directory that contains the expected data files.
    Supports Colab layout (/content/Recommendation_engine_group), a data/ folder
    with real_uploaded and synthetic subfolders, or a flat upload folder.
    """
    candidates = [
        Path('/content/Recommendation_engine_group'),
        Path('/content'),
        Path('/mnt/user-data/uploads'),
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path('/home/claude'),
    ]
    for base in candidates:
        if not base.exists():
            continue
        has_structured = (base / 'data').exists() and (
            (base / 'data' / 'real').exists()
            or (base / 'data' / 'real_uploaded').exists()
            or (base / 'data' / 'synthetic').exists()
        )
        has_flat = (base / 'players_master_with_market_values.csv').exists() or (
            base / 'players_master_synthetic_augmented.csv'
        ).exists()
        if has_structured or has_flat:
            return base
    raise FileNotFoundError('Could not locate project root with the expected data files.')

ROOT = locate_project_root()
print('Using project root:', ROOT)

def read_first_existing(paths, **kwargs):
    last_err = None
    for p in paths:
        p = Path(p)
        if p.exists():
            try:
                return pd.read_csv(p, **kwargs)
            except Exception as e:
                last_err = e
    if last_err is not None:
        raise last_err
    raise FileNotFoundError(f'None of the candidate paths exist: {paths}')

def load_real_tables(root=ROOT):
    players = read_first_existing([
        root / 'data' / 'real_uploaded' / 'players_master_with_market_values.csv',
        root / 'data' / 'real' / 'players_master_with_market_values.csv.gz',
        root / 'players_master_with_market_values.csv',
    ], low_memory=False)
    teams = read_first_existing([
        root / 'data' / 'real_uploaded' / 'teams_master_with_market_values.csv',
        root / 'data' / 'real' / 'teams_master_with_market_values.csv.gz',
        root / 'teams_master_with_market_values.csv',
    ], low_memory=False)
    return players, teams

def load_synthetic_tables(root=ROOT):
    players_aug = read_first_existing([
        root / 'data' / 'synthetic' / 'players_master_synthetic_augmented.csv',
        root / 'players_master_synthetic_augmented.csv',
    ], low_memory=False)
    teams_aug = read_first_existing([
        root / 'data' / 'synthetic' / 'teams_master_synthetic_augmented.csv',
        root / 'teams_master_synthetic_augmented.csv',
    ], low_memory=False)
    return players_aug, teams_aug

TRAIN_SEASONS = ['2018-19','2019-20','2020-21','2021-22']
VAL_SEASON = '2022-23'
TEST_SEASON = '2023-24'
PREV_SEASON = {'2019-20':'2018-19','2020-21':'2019-20','2021-22':'2020-21','2022-23':'2021-22','2023-24':'2022-23'}

PER90_MAP = {
    'misc__Performance__Int':'interceptions_p90',
    'misc__Performance__TklW':'tackles_won_p90',
    'misc__Performance__Fls':'fouls_committed_p90',
    'misc__Performance__Fld':'fouls_won_p90',
    'misc__Performance__Crs':'crosses_p90',
    'misc__Performance__Off':'offsides_p90',
}

REAL_SCORE_FEATURES = [
    'standard__Per 90 Minutes__G-PK',
    'standard__Per 90 Minutes__Ast',
    'shooting__Standard__Sh/90',
    'shooting__Standard__SoT/90',
    'shooting__Standard__G/Sh',
    'shooting__Standard__SoT%',
    'interceptions_p90',
    'tackles_won_p90',
    'crosses_p90',
    'fouls_won_p90',
    'offsides_p90',
    'playing_time__Team Success__+/-90',
    'playing_time__Team Success__On-Off',
    'playing_time__Playing Time__Min%',
]

CONTENT_REAL_VECTOR = [
    'standard__Per 90 Minutes__G-PK_z',
    'standard__Per 90 Minutes__Ast_z',
    'shooting__Standard__Sh/90_z',
    'shooting__Standard__SoT/90_z',
    'shooting__Standard__G/Sh_z',
    'shooting__Standard__SoT%_z',
    'interceptions_p90_z',
    'tackles_won_p90_z',
    'crosses_p90_z',
    'fouls_won_p90_z',
    'playing_time__Team Success__+/-90_z',
    'playing_time__Team Success__On-Off_z',
    'playing_time__Playing Time__Min%_z',
]

SYNTH_VECTOR_COLS = [
    'syn_trait_finishing','syn_trait_creativity','syn_trait_progression','syn_trait_ball_carrying',
    'syn_trait_defensive_intensity','syn_trait_press_resistance','syn_trait_aerial_physicality',
    'syn_trait_offball_threat','syn_xg_p90','syn_xa_p90','syn_xgi_p90','syn_progressive_passes_p90',
    'syn_progressive_carries_p90','syn_key_passes_p90','syn_tackles_won_p90','syn_interceptions_p90',
    'syn_pressures_applied_p90','syn_ball_retention_pct','syn_availability_pct'
]

def _primary_position(pos):
    if pd.isna(pos):
        return 'UNK'
    return str(pos).split(',')[0].strip()

def add_position_columns(players):
    out = players.copy()
    out['pos_primary'] = out['pos'].apply(_primary_position)
    out['pos_family'] = out['pos_primary'].map({'GK':'GK','DF':'DF','MF':'MF','FW':'FW'}).fillna('UNK')
    return out

def add_per90_columns(players):
    out = add_position_columns(players)
    for raw_col, new_col in PER90_MAP.items():
        if raw_col in out.columns:
            out[new_col] = out[raw_col] / out['nineties'].clip(lower=0.1)
        else:
            out[new_col] = np.nan
    return out

def filtered_players(players):
    out = add_per90_columns(players)
    outfield = out['pos_family'].isin(['DF','MF','FW']) & (out['minutes_played'] >= 600)
    keepers = (out['pos_family'] == 'GK') & (out['minutes_played'] >= 900)
    return out.loc[outfield | keepers].copy().reset_index(drop=True)

def safe_group_zscore(df, col, group_cols):
    grouped = df.groupby(group_cols)[col]
    mean = grouped.transform('mean')
    std = grouped.transform('std').replace(0, np.nan)
    z = (df[col] - mean) / std
    return z.replace([np.inf,-np.inf], np.nan).fillna(0.0)

def infer_role_subtype(players_f):
    out = players_f.copy()
    out['creator_proxy'] = out['standard__Per 90 Minutes__Ast_z'] + 0.7*out['crosses_p90_z'] + 0.4*out['fouls_won_p90_z']
    out['defender_proxy'] = out['interceptions_p90_z'] + 0.9*out['tackles_won_p90_z'] - 0.2*out['crosses_p90_z']
    out['striker_proxy'] = out['shooting__Standard__Sh/90_z'] + 0.9*out['standard__Per 90 Minutes__G-PK_z'] + 0.5*out['offsides_p90_z']
    out['wing_proxy'] = out['standard__Per 90 Minutes__Ast_z'] + 0.8*out['crosses_p90_z'] + 0.4*out['fouls_won_p90_z']
    role = []
    for _, row in out.iterrows():
        fam = row['pos_family']
        if fam == 'GK':
            role.append('Goalkeeper')
        elif fam == 'DF':
            role.append('Full-back / Wing-back' if (row['crosses_p90_z'] + row['standard__Per 90 Minutes__Ast_z']) > 0.3 else 'Centre-back')
        elif fam == 'MF':
            if row['creator_proxy'] - row['defender_proxy'] > 0.45:
                role.append('Creator / Advanced MF')
            elif row['defender_proxy'] - row['creator_proxy'] > 0.45:
                role.append('Ball-winning / Holding MF')
            else:
                role.append('Box-to-box / Hybrid MF')
        elif fam == 'FW':
            if row['striker_proxy'] - row['wing_proxy'] > 0.5:
                role.append('Striker')
            elif row['wing_proxy'] - row['striker_proxy'] > 0.2:
                role.append('Winger / Support Forward')
            else:
                role.append('Second Striker / Hybrid Forward')
        else:
            role.append('Other')
    out['role_subtype'] = role
    return out

def build_real_scoring_table(players, weights_override=None):
    """Build the scored player-season table.
    weights_override lets the sensitivity analysis perturb the composite without
    duplicating the whole function. Keys expected: 'FW', 'MF', 'DF' each mapped to
    a dict of {feature_z_col: weight}.
    """
    out = filtered_players(players)
    for col in REAL_SCORE_FEATURES:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = out.groupby(['league','season_label','pos_family'])[col].transform(lambda s: s.fillna(s.median()))
        out[col] = out[col].fillna(out[col].median())
        out[f'{col}_z'] = safe_group_zscore(out, col, ['season_label','league','pos_family'])
    out = infer_role_subtype(out)

    mv = out['tm_market_value_eur_resolved'].fillna(out['tm_market_value_eur_resolved'].median())
    out['log_market_value'] = np.log1p(mv)
    out['log_market_value_z'] = safe_group_zscore(out, 'log_market_value', ['season_label','pos_family'])
    out['age_curve'] = np.exp(-((out['age'] - 26.0) ** 2) / (2 * 5.0**2))

    default_weights = {
        'FW': {
            'standard__Per 90 Minutes__G-PK_z': 0.36,
            'standard__Per 90 Minutes__Ast_z':  0.18,
            'shooting__Standard__SoT/90_z':     0.18,
            'shooting__Standard__G/Sh_z':       0.10,
            'fouls_won_p90_z':                  0.08,
            'playing_time__Team Success__+/-90_z': 0.10,
        },
        'MF': {
            'standard__Per 90 Minutes__Ast_z':  0.24,
            'standard__Per 90 Minutes__G-PK_z': 0.16,
            'interceptions_p90_z':              0.20,
            'tackles_won_p90_z':                0.16,
            'fouls_won_p90_z':                  0.10,
            'crosses_p90_z':                    0.06,
            'playing_time__Team Success__+/-90_z': 0.08,
        },
        'DF': {
            'interceptions_p90_z':              0.32,
            'tackles_won_p90_z':                0.28,
            'crosses_p90_z':                    0.10,
            'playing_time__Team Success__+/-90_z': 0.15,
            'playing_time__Team Success__On-Off_z': 0.15,
        },
    }
    weights = weights_override if weights_override is not None else default_weights

    out['real_quality_score'] = 0.0
    for fam, ws in weights.items():
        mask = out['pos_family'] == fam
        if not mask.any():
            continue
        score = np.zeros(mask.sum())
        for col, w in ws.items():
            if col in out.columns:
                score = score + w * out.loc[mask, col].to_numpy()
        out.loc[mask, 'real_quality_score'] = score

    gk = out['pos_family'] == 'GK'
    if gk.any():
        for col in ['keeper__Performance__Save%','keeper__Performance__CS%','keeper__Performance__GA90']:
            if col in out.columns:
                out[col] = out.groupby(['league','season_label'])[col].transform(lambda s: s.fillna(s.median()))
                out[col] = out[col].fillna(out[col].median())
                out[f'{col}_z'] = safe_group_zscore(out, col, ['season_label','league'])
            else:
                out[f'{col}_z'] = 0.0
        out.loc[gk, 'real_quality_score'] = (
            0.40*out.loc[gk,'keeper__Performance__Save%_z'] +
            0.30*out.loc[gk,'keeper__Performance__CS%_z'] -
            0.30*out.loc[gk,'keeper__Performance__GA90_z']
        )

    out['value_adjusted_score'] = out['real_quality_score'] - 0.22*out['log_market_value_z']
    out['trajectory_adjusted_score'] = out['real_quality_score'] * (0.6 + 0.4*out['age_curve'])
    return out

def top_nonpersonalized(players_scored, season=TEST_SEASON, pos_family='FW', score_col='real_quality_score', top_n=10):
    cols = ['player','team','age','pos_family','role_subtype','tm_market_value_eur_resolved',score_col]
    return (
        players_scored[(players_scored['season_label']==season) & (players_scored['pos_family']==pos_family)]
        .sort_values(score_col, ascending=False)[cols]
        .rename(columns={'tm_market_value_eur_resolved':'market_value_eur'})
        .head(top_n)
        .reset_index(drop=True)
    )

def read_synthetic_players(root=ROOT):
    return read_first_existing([
        root / 'data' / 'synthetic' / 'players_master_synthetic_augmented.csv',
        root / 'players_master_synthetic_augmented.csv',
    ], low_memory=False)

def read_synthetic_teams(root=ROOT):
    return read_first_existing([
        root / 'data' / 'synthetic' / 'teams_master_synthetic_augmented.csv',
        root / 'teams_master_synthetic_augmented.csv',
    ], low_memory=False)

# Paired style axes. Order matters: player pref column maps to the corresponding team latent column.
STYLE_AXES = [
    ('syn_pref_possession',  'syn_latent_possession_orientation'),
    ('syn_pref_directness',  'syn_latent_directness'),
    ('syn_pref_pressing',    'syn_latent_pressing_intensity'),
    ('syn_pref_width',       'syn_latent_width_crossing'),
    ('syn_pref_tempo',       'syn_latent_tempo'),
    ('syn_pref_territorial', 'syn_latent_territorial_dominance'),
]

CONTRIB_COLS = [
    'syn_contrib_box_threat',
    'syn_contrib_creation',
    'syn_contrib_progression',
    'syn_contrib_pressing',
    'syn_contrib_aerial',
]

def style_fit_score(player_row, team_row):
    '''Element-wise 1 - mean(|player_pref - team_latent|) across paired style axes.
    Higher is a better stylistic match. Both vectors are in [0, 1].'''
    diffs = []
    for p_col, t_col in STYLE_AXES:
        p = player_row.get(p_col, np.nan)
        t = team_row.get(t_col, np.nan)
        if pd.isna(p) or pd.isna(t):
            continue
        diffs.append(abs(p - t))
    if not diffs:
        return np.nan
    return 1.0 - float(np.mean(diffs))

def need_fit_score(player_row, team_need_vector):
    '''Dot product of the player's contribution vector with the team's need direction.
    team_need_vector is the normalised gap between a top-quartile peer profile and
    the team's current squad contribution profile at the target position.'''
    vec = np.array([player_row.get(c, 0.0) for c in CONTRIB_COLS], dtype=float)
    if np.all(np.isnan(vec)):
        return np.nan
    vec = np.nan_to_num(vec, nan=0.0)
    return float(np.dot(vec, team_need_vector))

def budget_fit_score(player_market_value_eur, team_budget_band, sigma_log=1.0):
    '''Gaussian decay around the team's budget band.
    team_budget_band is the synthetic layer's normalised budget score (0-100).
    We first map the player's log market value into a comparable 0-100 scale
    using a min-max over the season, then take exp(-diff^2 / (2*sigma^2)).'''
    if pd.isna(player_market_value_eur) or pd.isna(team_budget_band):
        return np.nan
    log_mv = np.log1p(player_market_value_eur)
    # Static reference scale: log(1+euro) across 100k to 200M roughly spans 11.5 to 19.1.
    # Map to 0-100 linearly clipped.
    lo, hi = 11.5, 19.1
    player_band = 100.0 * (np.clip(log_mv, lo, hi) - lo) / (hi - lo)
    diff = (player_band - team_budget_band) / 100.0
    return float(np.exp(-(diff ** 2) / (2 * (sigma_log ** 2))))

def compute_team_need_vector(players_aug_season, team, pos_family, peer_percentile=0.75):
    '''Return a unit-norm vector pointing from the team's current contribution
    profile towards the top-quartile peer profile at the same position.'''
    at_team = players_aug_season[(players_aug_season['team']==team) &
                                  (players_aug_season['pos_family']==pos_family)]
    peers = players_aug_season[players_aug_season['pos_family']==pos_family]
    if at_team.empty or peers.empty:
        return np.zeros(len(CONTRIB_COLS))
    team_profile = at_team[CONTRIB_COLS].mean(skipna=True).fillna(0.0).to_numpy(dtype=float)
    cutoff = peers['minutes_played'].quantile(peer_percentile)
    top = peers[peers['minutes_played'] >= cutoff]
    if top.empty:
        return np.zeros(len(CONTRIB_COLS))
    peer_profile = top[CONTRIB_COLS].mean(skipna=True).fillna(0.0).to_numpy(dtype=float)
    gap = peer_profile - team_profile
    norm = np.linalg.norm(gap)
    return gap if norm == 0 else gap / norm

def context_score(player_row, team_row, team_need_vector,
                  w_style=0.35, w_need=0.30, w_budget=0.20, w_quality=0.15):
    '''Transparent hybrid score. All sub-scores sit in roughly [0, 1] or [-1, 1].
    Weights documented here are the single source of truth for the PoC layer.'''
    style = style_fit_score(player_row, team_row)
    need = need_fit_score(player_row, team_need_vector)
    budget = budget_fit_score(player_row.get('tm_market_value_eur_resolved', np.nan),
                              team_row.get('syn_budget_band_index', np.nan))
    quality = player_row.get('real_quality_score', 0.0)
    # Normalise quality into a [0, 1] band using a logistic so it is comparable in scale.
    quality_n = 1.0 / (1.0 + np.exp(-quality))

    # Any missing sub-score gets a neutral 0.5 so the composite still ranks candidates.
    style = 0.5 if pd.isna(style) else style
    need = 0.0 if pd.isna(need) else need
    budget = 0.5 if pd.isna(budget) else budget

    return (w_style * style + w_need * (0.5 + 0.5 * need)
            + w_budget * budget + w_quality * quality_n), dict(
                style_fit_poc=style, need_fit_poc=need,
                budget_fit_poc=budget, quality=quality_n)

def context_shortlist(players_aug, teams_aug, players_scored, team, target_season=TEST_SEASON,
                      pos_family='MF', top_n=10):
    '''Build a transparent context-aware shortlist for one club.'''
    prev = PREV_SEASON[target_season]
    # Build candidate pool from the previous season, excluding players already at the club.
    pool = (players_aug[players_aug['season_label']==prev]
            .sort_values(['player_key','minutes_played'], ascending=[True, False])
            .drop_duplicates('player_key').copy())
    pool = add_position_columns(pool)
    pool = pool[pool['pos_family']==pos_family]
    prev_team_players = set(players_aug.loc[
        (players_aug['season_label']==prev) & (players_aug['team']==team), 'player_key'])
    pool = pool[~pool['player_key'].isin(prev_team_players)].copy()

    # Attach real_quality_score from the scored table where we have it.
    scored = players_scored[['player_key','season_label','real_quality_score']].copy()
    pool = pool.merge(scored, on=['player_key','season_label'], how='left')
    pool['real_quality_score'] = pool['real_quality_score'].fillna(0.0)

    # Team row and need vector come from the previous season snapshot.
    team_row = teams_aug[(teams_aug['team']==team) & (teams_aug['season_label']==prev)]
    if team_row.empty:
        return pd.DataFrame()
    team_row = team_row.iloc[0]
    prev_season_players = players_aug[players_aug['season_label']==prev].copy()
    prev_season_players = add_position_columns(prev_season_players)
    need_vec = compute_team_need_vector(prev_season_players, team, pos_family)

    rows = []
    for _, r in pool.iterrows():
        score, parts = context_score(r, team_row, need_vec)
        rows.append({
            'player': r.get('player'),
            'team': r.get('team'),
            'pos_family': pos_family,
            'role_archetype': r.get('role_archetype', r.get('role_subtype', 'UNK')),
            'market_value_eur': r.get('tm_market_value_eur_resolved', np.nan),
            'style_fit_poc': parts['style_fit_poc'],
            'need_fit_poc': parts['need_fit_poc'],
            'budget_fit_poc': parts['budget_fit_poc'],
            'quality_logistic': parts['quality'],
            'context_syn_poc_score': score,
            'actual_arrival': bool(players_aug[
                (players_aug['player_key']==r['player_key']) &
                (players_aug['team']==team) &
                (players_aug['season_label']==target_season)]
                ['inferred_transfer_arrival_this_team'].any()) if (
                'inferred_transfer_arrival_this_team' in players_aug.columns) else False,
        })
    out = pd.DataFrame(rows).sort_values('context_syn_poc_score', ascending=False).head(top_n).reset_index(drop=True)
    out['fit_explanation'] = out.apply(lambda r: (
        f"style={r.style_fit_poc:.2f}, need={r.need_fit_poc:+.2f}, "
        f"budget={r.budget_fit_poc:.2f}, quality={r.quality_logistic:.2f}"), axis=1)
    return out

players, teams = load_real_tables(ROOT)
players_aug = read_synthetic_players(ROOT)
teams_aug = read_synthetic_teams(ROOT)
players_scored = build_real_scoring_table(players)

overview = pd.DataFrame({
    'table': ['players (real)','players (synthetic-augmented)','teams (synthetic-augmented)','players_scored'],
    'rows': [len(players), len(players_aug), len(teams_aug), len(players_scored)],
    'columns': [players.shape[1], players_aug.shape[1], teams_aug.shape[1], players_scored.shape[1]],
})
display(overview)


Using project root: /mnt/user-data/uploads


,table,rows,columns
0,players (real),16873,152
1,players (synthetic-augmented),16873,248
2,teams (synthetic-augmented),586,156
3,players_scored,11093,188


## Evidence: style axes and contribution vector

The six paired style axes and five contribution dimensions that feed the scoring formula. Showing them explicitly so the synthetic layer is not a black box.


In [2]:
style_axes_df = pd.DataFrame(STYLE_AXES, columns=['player_preference_col', 'team_latent_col'])
display(style_axes_df)

contrib_df = pd.DataFrame({'contribution_col': CONTRIB_COLS})
display(contrib_df)

# Coverage of synthetic features.
syn_coverage = pd.DataFrame({
    'column': [c for _, c in STYLE_AXES] + ['syn_budget_band_index', 'syn_tactical_archetype'],
    'teams_with_value': [teams_aug[c].notna().sum() for _, c in STYLE_AXES] +
                        [teams_aug['syn_budget_band_index'].notna().sum(),
                         teams_aug['syn_tactical_archetype'].notna().sum()],
})
display(syn_coverage)

print('Player synthetic pref coverage (2023-24 rows with complete pref vector):')
prefs = [p for p, _ in STYLE_AXES]
complete = players_aug[players_aug['season_label']==TEST_SEASON][prefs].notna().all(axis=1).sum()
total = (players_aug['season_label']==TEST_SEASON).sum()
print(f'  {complete} of {total} rows ({100*complete/total:.1f}%)')


,player_preference_col,team_latent_col
0,syn_pref_possession,syn_latent_possession_orientation
1,syn_pref_directness,syn_latent_directness
2,syn_pref_pressing,syn_latent_pressing_intensity
3,syn_pref_width,syn_latent_width_crossing
4,syn_pref_tempo,syn_latent_tempo
5,syn_pref_territorial,syn_latent_territorial_dominance


,contribution_col
0,syn_contrib_box_threat
1,syn_contrib_creation
2,syn_contrib_progression
3,syn_contrib_pressing
4,syn_contrib_aerial


,column,teams_with_value
0,syn_latent_possession_orientation,586
1,syn_latent_directness,586
2,syn_latent_pressing_intensity,586
3,syn_latent_width_crossing,586
4,syn_latent_tempo,586
5,syn_latent_territorial_dominance,586
6,syn_budget_band_index,586
7,syn_tactical_archetype,586


Player synthetic pref coverage (2023-24 rows with complete pref vector):
  1830 of 2852 rows (64.2%)


## Team tactical archetypes in 2023-24

The synthetic layer clusters teams into four broad archetypes. Distribution below.


In [3]:
team_archetypes = (teams_aug[teams_aug['season_label'] == TEST_SEASON]
                   .groupby('syn_tactical_archetype').size()
                   .reset_index(name='teams').sort_values('teams', ascending=False))
display(team_archetypes)

# Sample teams per archetype for context.
sample = (teams_aug[teams_aug['season_label'] == TEST_SEASON]
          .groupby('syn_tactical_archetype').apply(lambda g: g.nlargest(3, 'syn_budget_band_index')[['team','syn_budget_band_index']])
          .reset_index(level=0).reset_index(drop=True))
display(sample)


,syn_tactical_archetype,teams
0,aggressive_vertical_press,30
3,reactive_midblock_transition,25
2,patient_possession_control,22
1,dominant_wide_control,19


,syn_tactical_archetype,team,syn_budget_band_index
0,aggressive_vertical_press,West Ham United,62.879781
1,aggressive_vertical_press,Fulham,60.733862
2,aggressive_vertical_press,Everton,59.139484
3,dominant_wide_control,Manchester City,80.785532
4,dominant_wide_control,Barcelona,71.382669
5,dominant_wide_control,Milan,70.756402
6,patient_possession_control,Chelsea,74.274744
7,patient_possession_control,Manchester Utd,72.846019
8,patient_possession_control,Aston Villa,71.827343
9,reactive_midblock_transition,Crystal Palace,65.593063


## Audit: actual arrivals versus non-arrivals

For every 2023-24 arrival we know of, compare the four sub-scores against the non-arriving candidate pool at the same club and position. A positive delta means the layer ranks actual arrivals above non-arrivals, which is the PoC uplift signal. This is the replacement for the old `synthetic_generation_audit.json` display.


In [4]:
# Rebuild the audit in-notebook from the synthetic data we have.
target = players_aug[players_aug['season_label']==TEST_SEASON].copy()
target = add_position_columns(target)
target['is_arrival'] = target['inferred_transfer_arrival_this_team'].fillna(False).astype(bool)

# Attach the previous-season team latent vector to every (team, season) pair.
prev = PREV_SEASON[TEST_SEASON]
team_snap = teams_aug[teams_aug['season_label']==prev].copy()
target = target.merge(team_snap, on='team', how='left', suffixes=('','_team'))

# Compute per-row style fit against the joined team's latent vector.
def row_style(row):
    diffs = []
    for p_col, t_col in STYLE_AXES:
        p, t = row.get(p_col), row.get(t_col)
        if pd.notna(p) and pd.notna(t):
            diffs.append(abs(p - t))
    return 1.0 - np.mean(diffs) if diffs else np.nan

target['style_fit_audit'] = target.apply(row_style, axis=1)
target['budget_fit_audit'] = target.apply(
    lambda r: budget_fit_score(r.get('tm_market_value_eur_resolved'),
                               r.get('syn_budget_band_index')), axis=1)

audit = (target.groupby(['pos_family','is_arrival'])[['style_fit_audit','budget_fit_audit']]
         .mean().round(3).reset_index())
display(audit)

# Delta version: arrivals minus non-arrivals per position.
delta = (audit.set_index(['pos_family','is_arrival'])
         .unstack('is_arrival'))
for metric in ['style_fit_audit','budget_fit_audit']:
    delta[(metric, 'delta_true_minus_false')] = delta[(metric, True)] - delta[(metric, False)]
display(delta.round(3))


,pos_family,is_arrival,style_fit_audit,budget_fit_audit
0,DF,False,0.849,0.987
1,DF,True,0.854,0.988
2,FW,False,0.853,0.982
3,FW,True,0.851,0.990
4,GK,False,0.839,0.976
5,GK,True,0.850,0.990
6,MF,False,0.855,0.987
7,MF,True,0.856,0.993


style_fit_audit        budget_fit_audit         \
is_arrival           False   True            False   True   
pos_family                                                  
DF                   0.849  0.854            0.987  0.988   
FW                   0.853  0.851            0.982  0.990   
GK                   0.839  0.850            0.976  0.990   
MF                   0.855  0.856            0.987  0.993   

                  style_fit_audit       budget_fit_audit  
is_arrival delta_true_minus_false delta_true_minus_false  
pos_family                                                
DF                          0.005                  0.001  
FW                         -0.002                  0.008  
GK                          0.011                  0.014  
MF                          0.001                  0.006

## Arsenal context-aware shortlist (synthetic proof-of-concept)

Top-10 midfield candidates ranked by `context_syn_poc_score`, with explanation strings so each recommendation is auditable rather than opaque.


In [5]:
# We need pos_family on players_aug for the shortlist helper.
players_aug_pos = add_position_columns(players_aug)
arsenal = context_shortlist(players_aug_pos, teams_aug, players_scored,
                            team='Arsenal', target_season=TEST_SEASON,
                            pos_family='MF', top_n=10)
display(arsenal[['player','team','pos_family','role_archetype','context_syn_poc_score',
                 'actual_arrival','style_fit_poc','need_fit_poc','budget_fit_poc','fit_explanation']])


,player,team,pos_family,role_archetype,context_syn_poc_score,actual_arrival,style_fit_poc,need_fit_poc,budget_fit_poc,fit_explanation
0,Aurélien Tchouaméni,Real Madrid,MF,ball_winning_midfielder,1.067211,False,0.786789,2.275061,0.982882,"style=0.79, need=+2.28, budget=0.98, quality=0.69"
1,Valentin Rongier,Marseille,MF,ball_winning_midfielder,1.056476,False,0.707077,2.520031,0.999968,"style=0.71, need=+2.52, budget=1.00, quality=0.54"
2,Morten Hjulmand,Lecce,MF,ball_winning_midfielder,1.020153,False,0.583139,2.460732,0.999101,"style=0.58, need=+2.46, budget=1.00, quality=0.65"
3,Conor Gallagher,Chelsea,MF,ball_winning_midfielder,1.014741,False,0.720600,2.138671,0.998360,"style=0.72, need=+2.14, budget=1.00, quality=0.61"
4,João Palhinha,Fulham,MF,ball_winning_midfielder,1.007821,False,0.672465,2.239756,0.996253,"style=0.67, need=+2.24, budget=1.00, quality=0.58"
5,Tommaso Pobega,Milan,MF,ball_winning_midfielder,1.003383,False,0.737083,2.129247,0.998676,"style=0.74, need=+2.13, budget=1.00, quality=0.51"
6,Geoffrey Kondogbia,Atlético Madrid,MF,ball_winning_midfielder,0.999681,False,0.711063,1.939066,0.998676,"style=0.71, need=+1.94, budget=1.00, quality=0.73"
7,Robin Gosens,Inter,MF,ball_winning_midfielder,0.998705,False,0.777386,1.905090,0.999101,"style=0.78, need=+1.91, budget=1.00, quality=0.61"
8,Thiago Mendes,Lyon,MF,ball_winning_midfielder,0.994316,False,0.682628,2.205978,0.982675,"style=0.68, need=+2.21, budget=0.98, quality=0.52"
9,Casemiro,Manchester Utd,MF,ball_winning_midfielder,0.993930,False,0.720939,1.984019,0.996253,"style=0.72, need=+1.98, budget=1.00, quality=0.63"


## Girona context-aware shortlist (synthetic proof-of-concept)

Contrast: Girona is a very different club profile from Arsenal, so the style and budget components should push the recommendation in a different direction.


In [6]:
girona = context_shortlist(players_aug_pos, teams_aug, players_scored,
                           team='Girona', target_season=TEST_SEASON,
                           pos_family='FW', top_n=10)
display(girona[['player','team','pos_family','role_archetype','context_syn_poc_score',
                'actual_arrival','style_fit_poc','need_fit_poc','budget_fit_poc','fit_explanation']])


,player,team,pos_family,role_archetype,context_syn_poc_score,actual_arrival,style_fit_poc,need_fit_poc,budget_fit_poc,fit_explanation
0,Domenico Berardi,Sassuolo,FW,wide_creator_forward,0.866428,False,0.873384,0.834369,0.952939,"style=0.87, need=+0.83, budget=0.95, quality=0.63"
1,Jack Grealish,Manchester City,FW,wide_creator_forward,0.851727,False,0.840212,0.956195,0.883256,"style=0.84, need=+0.96, budget=0.88, quality=0.58"
2,Amine Adli,Leverkusen,FW,wide_creator_forward,0.849637,False,0.824395,0.883656,0.948754,"style=0.82, need=+0.88, budget=0.95, quality=0.59"
3,Marco Asensio,Real Madrid,FW,wide_creator_forward,0.849113,False,0.798611,0.751036,0.939357,"style=0.80, need=+0.75, budget=0.94, quality=0.79"
4,Sebastian Polter,Schalke 04,FW,wide_creator_forward,0.843947,False,0.856922,0.729722,0.995094,"style=0.86, need=+0.73, budget=1.00, quality=0.57"
5,Nicola Sansone,Bologna,FW,wide_creator_forward,0.828695,False,0.862019,0.438447,0.999864,"style=0.86, need=+0.44, budget=1.00, quality=0.74"
6,Jacob Murphy,Newcastle United,FW,wide_creator_forward,0.824903,False,0.894387,0.656936,0.959787,"style=0.89, need=+0.66, budget=0.96, quality=0.48"
7,Mattia Zaccagni,Lazio,FW,wide_creator_forward,0.824078,False,0.809314,0.723681,0.931151,"style=0.81, need=+0.72, budget=0.93, quality=0.64"
8,Adam Hložek,Leverkusen,FW,wide_creator_forward,0.822883,False,0.833955,0.734297,0.952939,"style=0.83, need=+0.73, budget=0.95, quality=0.54"
9,Phil Foden,Manchester City,FW,wide_creator_forward,0.822449,False,0.856596,0.611749,0.860261,"style=0.86, need=+0.61, budget=0.86, quality=0.73"


## Role-level uplift summary in the test window

Mean `context_syn_poc_score` for actual arrivals versus non-arrivals across every club at every position, grouped by role. This replaces the opaque `role_uplift_summary_test` table. Positive `mean_pair_uplift` means the synthetic layer is separating arrivals from non-arrivals in the expected direction on the PoC data.


In [7]:
# Vectorised context score across every (team, candidate) pair in the test season.
# Pairs every player-season in the TEST season with the team row for the previous
# season, so the team context is the state at recommendation time.
prev = PREV_SEASON[TEST_SEASON]

# 1) Candidate table: one row per (player_key, team, season_label=TEST), with
#    real_quality_score attached from players_scored.
cand = players_aug_pos[players_aug_pos['season_label']==TEST_SEASON][
    ['player_key','player','team','pos_family','tm_market_value_eur_resolved',
     'inferred_transfer_arrival_this_team'] + [p for p,_ in STYLE_AXES] + CONTRIB_COLS].copy()
cand = cand.rename(columns={'inferred_transfer_arrival_this_team': 'is_arrival'})
cand['is_arrival'] = cand['is_arrival'].fillna(False).astype(bool)
cand = cand.merge(players_scored[['player_key','season_label','real_quality_score']]
                  .query("season_label==@TEST_SEASON"),
                  on=['player_key'], how='left').drop(columns=['season_label'])
cand['real_quality_score'] = cand['real_quality_score'].fillna(0.0)

# 2) Team context table: from previous season.
team_ctx = teams_aug[teams_aug['season_label']==prev][
    ['team','syn_budget_band_index'] + [t for _,t in STYLE_AXES]].copy()

# 3) Join on team to pair each candidate with its team's context.
pair = cand.merge(team_ctx, on='team', how='left')

# 4) Vectorised style fit: 1 - mean |pref - latent| across axes.
pref_cols = [p for p,_ in STYLE_AXES]
latent_cols = [t for _,t in STYLE_AXES]
P = pair[pref_cols].to_numpy(dtype=float)
T = pair[latent_cols].to_numpy(dtype=float)
mask = ~(np.isnan(P) | np.isnan(T))
abs_diff = np.where(mask, np.abs(P - T), 0.0)
axis_counts = mask.sum(axis=1)
style_fit = np.where(axis_counts > 0,
                     1.0 - abs_diff.sum(axis=1) / np.maximum(axis_counts, 1),
                     0.5)
pair['style_fit_poc'] = style_fit

# 5) Need fit: compute one need vector per (team, pos_family) in the prev season.
prev_pool = add_position_columns(players_aug[players_aug['season_label']==prev].copy())
need_lookup = {}
for (team_name, pf), grp in prev_pool.groupby(['team','pos_family']):
    if pf not in ['FW','MF','DF']:
        continue
    need_lookup[(team_name, pf)] = compute_team_need_vector(prev_pool, team_name, pf)

contrib = pair[CONTRIB_COLS].fillna(0.0).to_numpy(dtype=float)
need_vecs = np.stack([need_lookup.get((t, pf), np.zeros(len(CONTRIB_COLS)))
                      for t, pf in zip(pair['team'], pair['pos_family'])])
pair['need_fit_poc'] = (contrib * need_vecs).sum(axis=1)

# 6) Budget fit: Gaussian decay.
log_mv = np.log1p(pair['tm_market_value_eur_resolved'].fillna(0.0).to_numpy(dtype=float))
lo, hi = 11.5, 19.1
player_band = 100.0 * (np.clip(log_mv, lo, hi) - lo) / (hi - lo)
diff = (player_band - pair['syn_budget_band_index'].fillna(50.0).to_numpy(dtype=float)) / 100.0
pair['budget_fit_poc'] = np.exp(-(diff ** 2) / 2.0)

# 7) Quality via logistic.
pair['quality_logistic'] = 1.0 / (1.0 + np.exp(-pair['real_quality_score'].to_numpy(dtype=float)))

# 8) Composite.
pair['context_syn_poc_score'] = (
    0.35 * pair['style_fit_poc']
    + 0.30 * (0.5 + 0.5 * pair['need_fit_poc'])
    + 0.20 * pair['budget_fit_poc']
    + 0.15 * pair['quality_logistic']
)

# 9) Uplift: arrivals vs non-arrivals per club-position, then aggregate by role.
grp = pair.groupby(['team','pos_family','is_arrival'])['context_syn_poc_score'].mean().unstack('is_arrival')
grp = grp.dropna(subset=[True])  # keep only cells with at least one arrival
grp['mean_pair_uplift'] = grp[True] - grp[False]
grp = grp.reset_index()
role_uplift = (grp.groupby('pos_family')
               .agg(mean_pair_uplift=('mean_pair_uplift','mean'),
                    teams_observed=('team','nunique'),
                    total_cells=('team','count'))
               .round(4).reset_index()
               .sort_values('mean_pair_uplift', ascending=False))
display(role_uplift)
print(f'\nTotal club-position cells with at least one arrival: {len(grp)}')
print(f'Cells with positive uplift: {(grp.mean_pair_uplift > 0).sum()} '
      f'({100*(grp.mean_pair_uplift > 0).mean():.1f}%)')

# Save the pair table for the evaluation notebook. Not required but useful.
pair_minimal = pair[['player','team','pos_family','is_arrival','context_syn_poc_score',
                     'style_fit_poc','need_fit_poc','budget_fit_poc','quality_logistic']]
print('\nScored pair-table shape:', pair_minimal.shape)
display(pair_minimal.head(5))


,pos_family,mean_pair_uplift,teams_observed,total_cells
3,MF,0.0316,85,85
0,DF,0.0215,72,72
1,FW,0.0178,53,53
2,GK,0.0055,27,27



Total club-position cells with at least one arrival: 237
Cells with positive uplift: 135 (57.0%)

Scored pair-table shape: (2889, 9)


,player,team,pos_family,is_arrival,context_syn_poc_score,style_fit_poc,need_fit_poc,budget_fit_poc,quality_logistic
0,Aaron Ramsdale,Arsenal,GK,False,0.599938,0.500000,0.000000,0.999692,0.500000
1,Ben White,Arsenal,DF,False,0.820854,0.791631,0.786894,0.991771,0.515963
2,Bukayo Saka,Arsenal,FW,False,0.571704,0.818225,-0.931859,0.968872,0.542199
3,Cédric Soares,Arsenal,DF,False,0.590763,0.500000,0.000000,0.953815,0.500000
4,David Raya,Arsenal,GK,True,0.774851,0.792084,0.000000,0.997617,0.987318


## Transparency note

The shortlist tables above belong to the synthetic proof-of-concept and scenario-modelling track. They are useful for richer tactical explanation and for shortlist storytelling, but they are not presented as unbiased real-world generalisation. The group brief requires multiple metric families and an explicit business/context layer; this notebook provides the modelling side. The honest benchmark comparison is kept in `05_evaluation.ipynb`, which uses only real FBref features and real inferred transfer arrivals as ground truth.

Everything above was computed in this notebook from the raw synthetic CSVs. No pre-computed shortlist or audit JSON was consumed. The scoring function, the weights and the need vector construction are defined in the setup cell and can be inspected line by line.
